# Full-data training: Amazon ML Challenge 2026 (v3 pipeline)

Runtime: **CPU, High-RAM (≥ 32 GB)**. No GPU is needed.

1. Upload `6ab10eb3b23ba_student_resource.zip` to Google Drive at `MyDrive/amlc/`.
2. Run the cells top to bottom. If the session disconnects, rerun from cell 1: finished stages are cached in `/content/work` and skipped, as long as the VM is still the same.

See `REPORT_v3.md` section 5 for details.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!free -g && nproc

In [ ]:
# 1) code
!rm -rf /content/amazon_ml && git clone -q -b v3-scalable-pipeline https://github.com/YatharthJangid/amazon_ml.git /content/amazon_ml
!pip install -q -r /content/amazon_ml/code/business_entity_resolution/requirements_v2.txt

In [ ]:
# 2) data onto the fast local disk (skip if already unpacked)
import os
DATA = '/content/6ab10eb3b23ba_student_resource/student_resource/dataset'
if not os.path.exists(DATA):
    !unzip -q /content/drive/MyDrive/amlc/6ab10eb3b23ba_student_resource.zip -d /content/
!ls -la {DATA}/train {DATA}/test

## 3) Run the pipeline

- `--train_s1 0` trains on **all** S1 entities. Use `600000` if memory gets tight.
- `--folds 0` skips CV and uses threshold 0.70 and 800 rounds (the CV optimum). Use `--folds 3` if there is time: it prints the OOF macro F0.5 for the documentation and re-tunes the threshold.

Expect ~2.5–3 h in total. Progress is logged with timestamps.

In [ ]:
%cd /content/amazon_ml/code/business_entity_resolution/src/v2
!python pipeline.py --data {DATA} --work /content/work --out /content/output --train_s1 0 --folds 0 all

In [ ]:
# 4) validate (official script)
!python /content/amazon_ml/utils/validate_submission.py --matching /content/output/matching_results.tsv --candidate /content/output/candidate_pairs.tsv --test-dir {DATA}/test --check-ids

In [ ]:
# 5) save results to Drive
!mkdir -p /content/drive/MyDrive/amlc/sub03
!cp /content/output/*.tsv /content/work/lgbm.txt /content/work/model_meta.json /content/drive/MyDrive/amlc/sub03/
!ls -la /content/drive/MyDrive/amlc/sub03
!cat /content/work/model_meta.json